# 05 - Estatística e sensibilidade (módulos 6 e 7)

Testes de hipótese com correção FDR (módulo 6) e ranking de sensibilidade por Cohen's d / Cliff's delta / permutation importance / SHAP (módulo 7).

In [1]:
import pandas as pd
from harmonic_signatures.config import PROCESSED_DIR
from harmonic_signatures.stats.group_tests import welch_ttest, mann_whitney, one_way_anova, sweep_channels
from harmonic_signatures.stats.confidence import confidence_interval_mean, correlation_with_ci
from harmonic_signatures.sensitivity.ranking import rank_channels_by_effect_size
from harmonic_signatures.sensitivity.importance import permutation_importance_ranking, shap_importance_ranking

harm_long = pd.read_parquet(PROCESSED_DIR / 'harmonicas_long.parquet')
harm_matrix = pd.read_parquet(PROCESSED_DIR / 'combined_matrix.parquet')

## Welch, Mann-Whitney e ANOVA — DIODO 3 vs máquina saudável, H5 Ia avg

In [2]:
print(welch_ttest(harm_long, 'DIODO 3', 'MÁQUINA BALANCEADA E ALINHADO', family='H', variable='I', phase='a', order=5, statistic='avg'))
print(mann_whitney(harm_long, 'DIODO 3', 'MÁQUINA BALANCEADA E ALINHADO', family='H', variable='I', phase='a', order=5, statistic='avg'))
print(one_way_anova(harm_long, family='H', variable='I', phase='a', order=5, statistic='avg'))

{'test': 'welch', 'label_a': 'DIODO 3', 'label_b': 'MÁQUINA BALANCEADA E ALINHADO', 'n_a': 648, 'n_b': 1440, 'statistic': 410.6203308105469, 'p_value': 0.0}


{'test': 'mann_whitney', 'label_a': 'DIODO 3', 'label_b': 'MÁQUINA BALANCEADA E ALINHADO', 'n_a': 648, 'n_b': 1440, 'statistic': 933120.0, 'p_value': 0.0}


{'test': 'anova', 'n_groups': 17, 'statistic': 130192.53125, 'p_value': 0.0}


## Sweep com correção FDR (Benjamini-Hochberg) — todos os canais H Ia avg, ANOVA entre os 17 rótulos

In [3]:
sweep = sweep_channels(harm_long, test='anova', channel_filter={'family': 'H', 'variable': 'I', 'statistic': 'avg'})
sweep[['family','variable','phase','order','p_value','p_adj','reject_h0']].head(10)

,family,variable,phase,order,p_value,p_adj,reject_h0
0,H,I,a,1,NaN,NaN,False
1,H,I,a,2,0.0,NaN,True
2,H,I,a,3,0.0,NaN,True
3,H,I,a,4,0.0,NaN,True
4,H,I,a,5,0.0,NaN,True
5,H,I,a,6,0.0,NaN,True
6,H,I,a,7,0.0,NaN,True
7,H,I,a,8,0.0,NaN,True
8,H,I,a,9,0.0,NaN,True
9,H,I,a,10,0.0,NaN,True


## Ranking de sensibilidade por Cliff's delta (DIODO 3 vs saudável)

In [4]:
ranking = rank_channels_by_effect_size(harm_long, 'DIODO 3', 'MÁQUINA BALANCEADA E ALINHADO', method='cliffs_delta', channel_filter={'family': 'H', 'statistic': 'avg'})
ranking[['family','variable','phase','order','effect_size','magnitude']].head(10)

,family,variable,phase,order,effect_size,magnitude
0,H,I,c,50,1.0,large
1,H,I,c,49,1.0,large
2,H,I,c,48,1.0,large
3,H,I,c,47,1.0,large
4,H,I,c,46,1.0,large
5,H,V,a,5,-1.0,large
6,H,I,c,45,1.0,large
7,H,I,c,29,1.0,large
8,H,I,c,28,1.0,large
9,H,I,c,27,1.0,large


## Permutation importance e SHAP (Random Forest, 17 classes)

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

feature_cols = [c for c in harm_matrix.columns if c.startswith('H') and c.endswith('_avg') and '_harm' not in c and '_inter' not in c][:40]
X = harm_matrix[feature_cols].fillna(0).to_numpy()
y = harm_matrix['label'].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1, max_depth=8).fit(X_train, y_train)
print('test accuracy:', clf.score(X_test, y_test))

test accuracy: 0.889933344727397


In [6]:
perm = permutation_importance_ranking(clf, X_test[:800], y_test[:800], feature_cols, n_repeats=5)
perm.head(10)

,feature,importance_mean,importance_std
0,H07_Ia_avg,0.02625,0.002500
1,H05_Va_avg,0.01600,0.006633
2,H05_Vc_avg,0.00300,0.002806
3,H05_Ia_avg,0.00200,0.004848
4,H06_Ib_avg,0.00150,0.004962
5,H06_Vc_avg,0.00100,0.000500
6,H03_Va_avg,0.00075,0.003921
7,H03_Vc_avg,0.00050,0.003122
8,H02_Ic_avg,0.00050,0.002179
9,H06_Vb_avg,0.00025,0.000500


In [7]:
shap_rank = shap_importance_ranking(clf, X_test, feature_cols, max_samples=150)
shap_rank.head(10)

 29%|======              | 740/2550 [00:11<00:26]       

 32%|======              | 811/2550 [00:12<00:25]       

 35%|=======             | 882/2550 [00:13<00:24]       

 37%|=======             | 952/2550 [00:14<00:23]       

 40%|========            | 1025/2550 [00:15<00:22]       

...

 91%|==================  | 2308/2550 [00:33<00:03]       

 93%|=================== | 2379/2550 [00:34<00:02]       

 96%|=================== | 2450/2550 [00:35<00:01]       

 99%|===================| 2522/2550 [00:36<00:00]       

,feature,mean_abs_shap
0,H07_Ia_avg,0.014227
1,H07_Ib_avg,0.011846
2,H05_Ib_avg,0.011422
3,H02_Ia_avg,0.010658
4,H04_Ib_avg,0.010501
5,H04_Ic_avg,0.009872
6,H06_Ib_avg,0.009074
7,H06_Ia_avg,0.009024
8,H06_Ic_avg,0.008844
9,H05_Va_avg,0.008410
